# Tahap 5 - Split Dataset untuk Modeling IndoBERT

Notebook ini mendokumentasikan proses pembagian dataset hasil labeling menjadi subset `train`, `validation`, dan `test` untuk project skripsi **Analisis Sentimen pada Aplikasi JakOne Mobile Menggunakan Metode IndoBERT**.

Tahap ini hanya melakukan split dataset. Tidak ada tokenisasi IndoBERT, training model, evaluasi model, visualisasi akhir, atau model output.

## Input dan Output

Input tahap ini adalah dataset hasil labeling InSet Lexicon:

```text
data/processed/jakone_reviews_labeled.csv
```

Output dataset final modeling:

```text
data/final/jakone_modeling_master.csv
```

Output ringkasan distribusi split:

```text
outputs/evaluation/split_distribution_summary.csv
```

## Strategi Split

Dataset dibagi dengan rasio:

```text
train      = 80%
validation = 10%
test       = 10%
```

Split dilakukan secara stratified berdasarkan kolom `label` agar proporsi `positif`, `negatif`, dan `netral` tetap seimbang di setiap subset.

In [ ]:
from pathlib import Path
import importlib.util

import pandas as pd

## Memuat Fungsi dari Script

Notebook ini menggunakan fungsi dari `src/06_split_dataset.py` agar hasilnya konsisten dengan script terminal.

In [ ]:
module_path = Path("src/06_split_dataset.py")
if not module_path.exists():
    module_path = Path("../src/06_split_dataset.py")

spec = importlib.util.spec_from_file_location("split_module", module_path)
split_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(split_module)

## Membaca dan Memvalidasi Dataset

Validasi memastikan file tersedia, kolom `clean_review` dan `label` ada, teks tidak kosong, label tidak kosong, dan label hanya berisi `positif`, `negatif`, atau `netral`.

In [ ]:
split_module.validate_input_file()
df = split_module.load_dataset()
split_module.validate_required_columns(df)
valid_df, total_initial, invalid_count = split_module.clean_invalid_rows(df)

print(f"Jumlah data awal: {total_initial}")
print(f"Jumlah data tidak valid yang dihapus: {invalid_count}")
print(f"Jumlah data valid: {len(valid_df)}")

valid_df["label"].value_counts().reindex(split_module.LABEL_ORDER, fill_value=0)

## Melakukan Stratified Split

Split pertama memisahkan 80% data sebagai `train` dan 20% sebagai data sementara. Split kedua membagi data sementara menjadi `validation` dan `test` dengan rasio 50:50.

In [ ]:
final_df = split_module.perform_stratified_split(valid_df)
final_df = split_module.order_output_columns(final_df)

final_df["split_set"].value_counts().reindex(split_module.SPLIT_ORDER, fill_value=0)

## Mengecek Distribusi Label per Split

Tabel berikut digunakan untuk memastikan proporsi label tetap seimbang setelah stratified split.

In [ ]:
count_table = pd.crosstab(final_df["split_set"], final_df["label"]).reindex(
    index=split_module.SPLIT_ORDER,
    columns=split_module.LABEL_ORDER,
    fill_value=0,
)
percentage_table = count_table.div(count_table.sum(axis=1), axis=0).mul(100).round(2)

display(count_table)
display(percentage_table)

## Ringkasan Distribusi Split

Ringkasan ini disimpan sebagai CSV untuk dokumentasi tahap split dataset.

In [ ]:
distribution_summary = split_module.build_distribution_summary(final_df)
distribution_summary

## Contoh Output Dataset Final

Kolom `split_set` menunjukkan subset modeling untuk setiap baris data.

In [ ]:
final_df[["review_id", "clean_review", "label", "split_set", "lexicon_score"]].head()

## Menyimpan Output

Dataset final modeling dan ringkasan distribusi split disimpan ke lokasi output yang sudah ditentukan.

In [ ]:
split_module.OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
split_module.SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)

final_df.to_csv(split_module.OUTPUT_PATH, index=False, encoding="utf-8-sig")
distribution_summary.to_csv(split_module.SUMMARY_PATH, index=False, encoding="utf-8-sig")

print(f"Dataset final: {split_module.OUTPUT_PATH}")
print(f"Ringkasan distribusi: {split_module.SUMMARY_PATH}")